## Materialized views — precomputed, auto-refreshed, queried like a table

A **materialized view (MV)** is a query result that Databricks computes once, stores as Delta, and **refreshes** on a declared cadence — or whenever you call `REFRESH MATERIALIZED VIEW`.

```sql
CREATE OR REPLACE MATERIALIZED VIEW fintech_dev.gold.customer_360
SCHEDULE EVERY 1 HOUR
AS
SELECT c.customer_id,
       c.city,
       COUNT(t.transaction_id)                AS txn_count_30d,
       SUM(t.amount)                           AS total_spend_30d,
       APPROX_COUNT_DISTINCT(t.merchant_name)  AS distinct_merchants_30d
FROM   fintech_dev.silver.customers c
LEFT JOIN fintech_dev.silver.card_transactions t
  ON   c.customer_id = t.customer_id
 AND   t.transaction_at >= current_date() - INTERVAL 30 DAYS
GROUP BY c.customer_id, c.city;
```

What an MV buys you:

- **Incremental refresh** — when base tables change, only affected rows recompute (when the query supports it).
- **Consistent reads** — like a table; not exposed to base-table mid-write inconsistency.
- **Cheap for BI** — analysts hit a precomputed result, not a 50-billion-row aggregation.
- **Declarative schedule** — the cadence lives *with the object*, not in a separate Lakeflow Jobs definition.

MVs are the **right answer** whenever a question describes *"a heavy aggregation that BI reads frequently, refreshed on a cadence."* Contrast with a plain view (recomputes every read) and a streaming table (continuous append, not periodic aggregation).
